# Faithfulness Eval — Colab runner

Current pipeline (no LLM judge anymore — validity and bid-think agreement are both plain arithmetic):
- **0.5B–32B** run **locally** on this Colab GPU via `transformers` (needs an A100, no `HF_TOKEN` required — public weights).
- **72B** runs via **OpenRouter** (`--provider openrouter`) since it does not fit locally even in 4-bit — needs `OPENROUTER_API_KEY`.

Before running: **Runtime → Change runtime type → A100 GPU + High-RAM**. Then open the Secrets tab (🔑) and add `OPENROUTER_API_KEY` (only needed for the 72B cell), toggling notebook access on.

## Setup

In [ ]:
import os
if not os.path.exists('faithfulness-study'):
    !git clone https://github.com/shema-boris/faithfulness-study.git
%cd faithfulness-study
!git pull

In [ ]:
!pip install -q openai

In [ ]:
import os
from google.colab import userdata

try:
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
except Exception:
    pass
print('OPENROUTER_API_KEY loaded:', bool(os.environ.get('OPENROUTER_API_KEY')))

## Confirm GPU
Should show an A100 with ~40GB. If not, go to Runtime → Change runtime type and select it before continuing.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Smoke test (no GPU, no API calls, no cost)
Confirms the pipeline parses the new structured Think format correctly before spending any real compute.

In [ ]:
!python faithfulness_eval.py --dry-run --n-scenarios 5 --n-rollouts 2 --verbose

## Install local-mode dependencies

In [ ]:
!pip install -q transformers torch accelerate bitsandbytes

## Quick live sanity check
Real model, real GPU, small n — confirms the structured Think format (constraint check / strategy / decided bid) actually gets followed before committing to the full ladder.

In [ ]:
!python faithfulness_eval.py --models "Qwen/Qwen2.5-0.5B-Instruct" --local --n-scenarios 3 --n-rollouts 1 --verbose

## Full local ladder — one model at a time
Each run appends to `results/raw_responses.csv` (the append-safety check will start fresh only if the schema changed). Run these in order; each cell loads its model, runs the full 20-scenario x 2-condition x N-rollout sweep, then frees GPU memory before you move to the next.

In [ ]:
!python faithfulness_eval.py --models "Qwen/Qwen2.5-0.5B-Instruct" --local --n-rollouts 2 --verbose

In [ ]:
!python faithfulness_eval.py --models "Qwen/Qwen2.5-1.5B-Instruct" --local --n-rollouts 2 --verbose

In [ ]:
!python faithfulness_eval.py --models "Qwen/Qwen2.5-3B-Instruct" --local --n-rollouts 2 --verbose

In [ ]:
!python faithfulness_eval.py --models "Qwen/Qwen2.5-7B-Instruct" --local --n-rollouts 2 --verbose

In [ ]:
!python faithfulness_eval.py --models "Qwen/Qwen2.5-14B-Instruct" --local --n-rollouts 2 --verbose

In [ ]:
# 32B needs 4-bit quantization to fit a 40GB A100 (fp16 would need ~64GB)
!python faithfulness_eval.py --models "Qwen/Qwen2.5-32B-Instruct" --local --load-in-4bit --n-rollouts 2 --verbose

## 72B — via OpenRouter
Too tight for a single 40GB A100 even in 4-bit with CPU offload (confirmed unworkable earlier). Needs `OPENROUTER_API_KEY` loaded above. Note the lowercase OpenRouter-style slug, different from the HF casing used above.

In [ ]:
!python faithfulness_eval.py --models "qwen/qwen-2.5-72b-instruct" --provider openrouter --n-rollouts 2 --verbose

## Inspect results

In [ ]:
import pandas as pd

summary = pd.read_csv('results/faithfulness_results.csv')
summary

In [ ]:
raw = pd.read_csv('results/raw_responses.csv')
raw.head(10)

## Push results to GitHub

In [ ]:
!git add -f results/raw_responses.csv results/faithfulness_results.csv
!git commit -m "Full ladder run with structured CoT + split prompts"
!git push